<center><h1>Hong_Brandon_HW7</h1></center>

Name: Brandon Hong
<br>
Github Username: bwhong
<br>
USC ID: 3091232124

## 1. Multi-class and Multi-Label Classification Using Support Vector Machines

Import packages

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC, LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, hamming_loss

from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

### (a) Download the Anuran Calls (MFCCs) Data Set

In [2]:
data = pd.read_csv(r'../data/Frogs_MFCCs.csv')
data.head()

,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,...,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species,RecordID
0,1.0,0.152936,-0.105586,0.200722,0.317201,0.260764,0.100945,-0.150063,-0.171128,0.124676,...,-0.108351,-0.077623,-0.009568,0.057684,0.118680,0.014038,Leptodactylidae,Adenomera,AdenomeraAndre,1
1,1.0,0.171534,-0.098975,0.268425,0.338672,0.268353,0.060835,-0.222475,-0.207693,0.170883,...,-0.090974,-0.056510,-0.035303,0.020140,0.082263,0.029056,Leptodactylidae,Adenomera,AdenomeraAndre,1
2,1.0,0.152317,-0.082973,0.287128,0.276014,0.189867,0.008714,-0.242234,-0.219153,0.232538,...,-0.050691,-0.023590,-0.066722,-0.025083,0.099108,0.077162,Leptodactylidae,Adenomera,AdenomeraAndre,1
3,1.0,0.224392,0.118985,0.329432,0.372088,0.361005,0.015501,-0.194347,-0.098181,0.270375,...,-0.136009,-0.177037,-0.130498,-0.054766,-0.018691,0.023954,Leptodactylidae,Adenomera,AdenomeraAndre,1
4,1.0,0.087817,-0.068345,0.306967,0.330923,0.249144,0.006884,-0.265423,-0.172700,0.266434,...,-0.048885,-0.053074,-0.088550,-0.031346,0.108610,0.079244,Leptodactylidae,Adenomera,AdenomeraAndre,1


In [3]:
data = data.drop(columns= {'RecordID'})
train_data, test_data = train_test_split(
    data , test_size=0.3, random_state=42
)
train_data.head(3)

,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,...,MFCCs_16,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species
2004,1.000000,0.327476,0.282440,0.572211,0.096509,-0.036705,-0.082174,0.146604,0.297338,-0.120902,...,0.178628,0.218024,-0.042083,-0.126511,-0.090279,0.096176,0.191697,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
1194,1.000000,0.232818,-0.144331,-0.078909,-0.072467,0.255380,0.450215,0.261033,-0.168462,-0.281445,...,-0.157579,-0.291979,-0.042932,-0.009021,-0.032679,0.023912,0.041407,Dendrobatidae,Ameerega,Ameeregatrivittata
5359,0.966443,0.455914,1.000000,0.541650,-0.412873,0.307099,0.235819,-0.064025,0.069728,-0.052552,...,-0.007981,-0.049428,-0.053166,-0.026728,0.029443,0.019032,-0.105221,Hylidae,Hypsiboas,HypsiboasCinerascens


In [4]:
test_data.head(3)

,MFCCs_ 1,MFCCs_ 2,MFCCs_ 3,MFCCs_ 4,MFCCs_ 5,MFCCs_ 6,MFCCs_ 7,MFCCs_ 8,MFCCs_ 9,MFCCs_10,...,MFCCs_16,MFCCs_17,MFCCs_18,MFCCs_19,MFCCs_20,MFCCs_21,MFCCs_22,Family,Genus,Species
3340,1.0,0.147506,0.014843,0.433466,0.169581,0.053135,-0.152020,-0.102427,0.153061,0.128628,...,0.016310,0.285756,0.166246,-0.061206,-0.191878,0.003221,0.255371,Leptodactylidae,Adenomera,AdenomeraHylaedactylus
5108,1.0,0.674174,0.767520,0.392177,0.035669,0.181548,-0.013677,0.187262,0.144679,-0.424524,...,0.103722,-0.157943,-0.048913,-0.016129,0.129200,0.011245,-0.275909,Hylidae,Hypsiboas,HypsiboasCinerascens
6524,1.0,-0.007777,0.363276,0.413224,0.196296,0.222389,0.097128,-0.155608,-0.035013,0.133689,...,-0.005594,0.125169,0.044882,-0.013309,-0.026086,-0.088825,-0.018968,Hylidae,Hypsiboas,HypsiboasCordobae


### (b) Train a classifier for each label

#### (i) Research

Exact Match is when we ignore partially correct and only look at exactly correct. However, it does not distinguish between completely incorrect and partially correct. Hamming Loss is the fraction of labels that are wrong across all instances and labels. This is different than exact match because it does not judge whether the whole prediction was correct or incorrect. 

To do exact match, we essentially compare each label, and even if one is incorrect, we mark the whole as incorrect. We would typically use accuracy_score, but it does not support multiclass-multioutput data. For hamming loss, we would calculate the fraction of labels predicted incorrectly on average. Hamming Score would just be 1 - hamming loss. 

#### (ii) Train a SVM for each of the labels

In [5]:
X_train = train_data.drop(columns = {'Family', 'Genus', 'Species'})
y_train = train_data[['Family', 'Genus', 'Species']]
y_train_family = train_data['Family']
y_train_genus = train_data['Genus']
y_train_species = train_data['Species']

In [6]:
X_test = test_data.drop(columns = {'Family', 'Genus', 'Species'})
y_test = test_data[['Family', 'Genus', 'Species']]
y_test_family = test_data['Family']
y_test_genus = test_data['Genus']
y_test_species = test_data['Species']

In [7]:
C_values = np.logspace(-3,6,10)

sigma_values = np.linspace(1, 5.0, 10)

param_grid = {"svm__estimator__C": C_values,
              "svm__estimator__gamma" : 1 / (2 * sigma_values **2)}

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm",  OneVsRestClassifier(SVC(kernel = "rbf")))])

I was originally do sigma values = np.linespace(0.1, 2.1, 0.1), but I saw that cross valdiation was consistently selecting the largest sigma value in my search grid. Therefore, I increased the upper boundary to 5 to properly explore the wdith of the Gaussian Kernel.

In [8]:
svm_family_model = GridSearchCV(svm_pipeline, param_grid, cv = StratifiedKFold(10, shuffle=True, random_state=42), scoring = "accuracy", n_jobs = -1)
svm_family_model.fit(X_train, y_train_family)

GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svm',
                                        OneVsRestClassifier(estimator=SVC()))]),
             n_jobs=-1,
             param_grid={'svm__estimator__C': array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03, 1.e+04,
       1.e+05, 1.e+06]),
                         'svm__estimator__gamma': array([0.5       , 0.23964497, 0.14013841, 0.09183673, 0.0648    ,
       0.04815696, 0.03719008, 0.02958364, 0.0240928 , 0.02      ])},
             scoring='accuracy')

In [9]:
best_family_model = svm_family_model.best_estimator_
y_pred_family = best_family_model.predict(X_test)

print("Best Parameters",svm_family_model.best_params_)
print("Best CV Accuracy", svm_family_model.best_score_)

Best Parameters {'svm__estimator__C': np.float64(10.0), 'svm__estimator__gamma': np.float64(0.0371900826446281)}
Best CV Accuracy 0.9922540155890056


In [10]:
svm_genus_model = GridSearchCV(svm_pipeline, param_grid, cv = StratifiedKFold(10, shuffle=True, random_state=42), scoring = "accuracy", n_jobs = -1)
svm_genus_model.fit(X_train, y_train_genus)

GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svm',
                                        OneVsRestClassifier(estimator=SVC()))]),
             n_jobs=-1,
             param_grid={'svm__estimator__C': array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03, 1.e+04,
       1.e+05, 1.e+06]),
                         'svm__estimator__gamma': array([0.5       , 0.23964497, 0.14013841, 0.09183673, 0.0648    ,
       0.04815696, 0.03719008, 0.02958364, 0.0240928 , 0.02      ])},
             scoring='accuracy')

In [11]:
best_genus_model = svm_genus_model.best_estimator_
y_pred_genus = best_genus_model.predict(X_test)

print("Best Parameters",svm_genus_model.best_params_)
print("Best CV Accuracy", svm_genus_model.best_score_)

Best Parameters {'svm__estimator__C': np.float64(10.0), 'svm__estimator__gamma': np.float64(0.0648)}
Best CV Accuracy 0.9918575846508251


In [12]:
svm_species_model = GridSearchCV(svm_pipeline, param_grid, cv = StratifiedKFold(10, shuffle=True, random_state=42), scoring = "accuracy", n_jobs = -1)
svm_species_model.fit(X_train, y_train_species)

GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svm',
                                        OneVsRestClassifier(estimator=SVC()))]),
             n_jobs=-1,
             param_grid={'svm__estimator__C': array([1.e-03, 1.e-02, 1.e-01, 1.e+00, 1.e+01, 1.e+02, 1.e+03, 1.e+04,
       1.e+05, 1.e+06]),
                         'svm__estimator__gamma': array([0.5       , 0.23964497, 0.14013841, 0.09183673, 0.0648    ,
       0.04815696, 0.03719008, 0.02958364, 0.0240928 , 0.02      ])},
             scoring='accuracy')

In [13]:
best_species_model = svm_species_model.best_estimator_
y_pred_species = best_species_model.predict(X_test)

print("Best Parameters",svm_species_model.best_params_)
print("Best CV Accuracy", svm_species_model.best_score_)

Best Parameters {'svm__estimator__C': np.float64(10.0), 'svm__estimator__gamma': np.float64(0.04815695600475624)}
Best CV Accuracy 0.9908667045346966


### Exact Score and Hamming Loss/Score

In [14]:
y_pred = pd.DataFrame({'Family': y_pred_family, 'Genus':y_pred_genus, 'Species' : y_pred_species})

exact_match_score = (y_test.values == y_pred.values).all(axis = 1).mean()
hamming_loss_value = (y_test.values != y_pred.values).mean()
hamming_score = 1 - hamming_loss_value

print("Exact Match Score:", exact_match_score)
print("Hamming Loss:", hamming_loss_value)
print("Hamming Score:", hamming_score)

Exact Match Score: 0.9856415006947661
Hamming Loss: 0.010653080129689671
Hamming Score: 0.9893469198703103


#### (iii) Repeat 1(b)ii with L1-penalized SVMs

In [15]:
C_values = np.logspace(-3,4,10)

param_grid = {"svm__estimator__C": C_values}

linear_svc_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("svm",  OneVsRestClassifier(LinearSVC(penalty='l1', dual=False, max_iter=30000)))])

In [16]:
linear_svc_family_model = GridSearchCV(linear_svc_pipeline, param_grid, cv = StratifiedKFold(10, shuffle=True, random_state=42), scoring = 'accuracy', n_jobs = -1)
linear_svc_family_model.fit(X_train, y_train_family)

GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svm',
                                        OneVsRestClassifier(estimator=LinearSVC(dual=False,
                                                                                max_iter=30000,
                                                                                penalty='l1')))]),
             n_jobs=-1,
             param_grid={'svm__estimator__C': array([1.00000000e-03, 5.99484250e-03, 3.59381366e-02, 2.15443469e-01,
       1.29154967e+00, 7.74263683e+00, 4.64158883e+01, 2.78255940e+02,
       1.66810054e+03, 1.00000000e+04])},
             scoring='accuracy')

In [17]:
best_linear_svc_family_model = linear_svc_family_model.best_estimator_
y_pred_family = best_linear_svc_family_model.predict(X_test)

print("Best Parameters",linear_svc_family_model.best_params_)
print("Best CV Accuracy", linear_svc_family_model.best_score_)

Best Parameters {'svm__estimator__C': np.float64(7.742636826811269)}
Best CV Accuracy 0.9412229006910916


In [18]:
linear_svc_genus_model = GridSearchCV(linear_svc_pipeline, param_grid, cv = StratifiedKFold(10, shuffle=True, random_state=42), scoring = 'accuracy', n_jobs = -1)
linear_svc_genus_model.fit(X_train, y_train_genus)

GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svm',
                                        OneVsRestClassifier(estimator=LinearSVC(dual=False,
                                                                                max_iter=30000,
                                                                                penalty='l1')))]),
             n_jobs=-1,
             param_grid={'svm__estimator__C': array([1.00000000e-03, 5.99484250e-03, 3.59381366e-02, 2.15443469e-01,
       1.29154967e+00, 7.74263683e+00, 4.64158883e+01, 2.78255940e+02,
       1.66810054e+03, 1.00000000e+04])},
             scoring='accuracy')

In [19]:
best_linear_svc_genus_model = linear_svc_genus_model.best_estimator_
y_pred_genus = best_linear_svc_genus_model.predict(X_test)

print("Best Parameters", linear_svc_genus_model.best_params_)
print("Best CV Accuracy", linear_svc_genus_model.best_score_)

Best Parameters {'svm__estimator__C': np.float64(278.2559402207126)}
Best CV Accuracy 0.9517498185490234


In [20]:
linear_svc_species_model = GridSearchCV(linear_svc_pipeline, param_grid, cv = StratifiedKFold(10, shuffle=True, random_state=42), scoring = 'accuracy', n_jobs = -1)
linear_svc_species_model.fit(X_train, y_train_species)

/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:1249: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(


GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('svm',
                                        OneVsRestClassifier(estimator=LinearSVC(dual=False,
                                                                                max_iter=30000,
                                                                                penalty='l1')))]),
             n_jobs=-1,
             param_grid={'svm__estimator__C': array([1.00000000e-03, 5.99484250e-03, 3.59381366e-02, 2.15443469e-01,
       1.29154967e+00, 7.74263683e+00, 4.64158883e+01, 2.78255940e+02,
       1.66810054e+03, 1.00000000e+04])},
             scoring='accuracy')

I've increased the max iterations and lowered the C parameter upper bound, but I still get some convergence errors. This is happening for higher C values. However, this model's best parameter is not near the upper bound regardless, so this error can be ignored.

In [21]:
best_linear_svc_species_model = linear_svc_species_model.best_estimator_
y_pred_species = best_linear_svc_species_model.predict(X_test)

print("Best Parameters", linear_svc_species_model.best_params_)
print("Best CV Accuracy", linear_svc_species_model.best_score_)

Best Parameters {'svm__estimator__C': np.float64(7.742636826811269)}
Best CV Accuracy 0.9606823345640443


### Exact Score and Hamming Loss/Score

In [22]:
y_pred = pd.DataFrame({'Family': y_pred_family, 'Genus':y_pred_genus, 'Species' : y_pred_species})

exact_match_score = (y_test.values == y_pred.values).all(axis = 1).mean()
hamming_loss_value = (y_test.values != y_pred.values).mean()
hamming_score = 1 - hamming_loss_value

print("Exact Match Score:", exact_match_score)
print("Hamming Loss:", hamming_loss_value)
print("Hamming Score:", hamming_score)

Exact Match Score: 0.9124594719777674
Hamming Loss: 0.05697081982399259
Hamming Score: 0.9430291801760075


#### (iv) Repeat 1(b)iii by using SMOTE or any other method for imbalance

In [23]:
C_values = np.logspace(-3,4,10)

param_grid = {"svm__estimator__C": C_values}

smote_linear_svc_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("svm",  OneVsRestClassifier(LinearSVC(penalty='l1', dual=False, max_iter=30000)))])

In [24]:
smote_linear_svc_family_model = GridSearchCV(smote_linear_svc_pipeline, param_grid, cv = StratifiedKFold(10, shuffle=True, random_state=42), scoring = 'accuracy', n_jobs = -1)
smote_linear_svc_family_model.fit(X_train, y_train_family)

GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote', SMOTE(random_state=42)),
                                       ('svm',
                                        OneVsRestClassifier(estimator=LinearSVC(dual=False,
                                                                                max_iter=30000,
                                                                                penalty='l1')))]),
             n_jobs=-1,
             param_grid={'svm__estimator__C': array([1.00000000e-03, 5.99484250e-03, 3.59381366e-02, 2.15443469e-01,
       1.29154967e+00, 7.74263683e+00, 4.64158883e+01, 2.78255940e+02,
       1.66810054e+03, 1.00000000e+04])},
             scoring='accuracy')

In [25]:
best_smote_linear_svc_family_model = smote_linear_svc_family_model.best_estimator_
y_pred_family = best_smote_linear_svc_family_model.predict(X_test)

print("Best Parameters",smote_linear_svc_family_model.best_params_)
print("Best CV Accuracy", smote_linear_svc_family_model.best_score_)

Best Parameters {'svm__estimator__C': np.float64(7.742636826811269)}
Best CV Accuracy 0.9231555113761873


In [26]:
smote_linear_svc_genus_model = GridSearchCV(smote_linear_svc_pipeline, param_grid, cv = StratifiedKFold(10, shuffle=True, random_state=42), scoring = 'accuracy', n_jobs = -1)
smote_linear_svc_genus_model.fit(X_train, y_train_genus)

GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote', SMOTE(random_state=42)),
                                       ('svm',
                                        OneVsRestClassifier(estimator=LinearSVC(dual=False,
                                                                                max_iter=30000,
                                                                                penalty='l1')))]),
             n_jobs=-1,
             param_grid={'svm__estimator__C': array([1.00000000e-03, 5.99484250e-03, 3.59381366e-02, 2.15443469e-01,
       1.29154967e+00, 7.74263683e+00, 4.64158883e+01, 2.78255940e+02,
       1.66810054e+03, 1.00000000e+04])},
             scoring='accuracy')

In [27]:
best_smote_linear_svc_genus_model = smote_linear_svc_genus_model.best_estimator_
y_pred_genus = best_smote_linear_svc_genus_model.predict(X_test)

print("Best Parameters",smote_linear_svc_genus_model.best_params_)
print("Best CV Accuracy", smote_linear_svc_genus_model.best_score_)

Best Parameters {'svm__estimator__C': np.float64(1.291549665014884)}
Best CV Accuracy 0.9128368676827922


In [28]:
smote_linear_svc_species_model = GridSearchCV(smote_linear_svc_pipeline, param_grid, cv = StratifiedKFold(10, shuffle=True, random_state=42), scoring = 'accuracy', n_jobs = -1)
smote_linear_svc_species_model.fit(X_train, y_train_species)

GridSearchCV(cv=StratifiedKFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('smote', SMOTE(random_state=42)),
                                       ('svm',
                                        OneVsRestClassifier(estimator=LinearSVC(dual=False,
                                                                                max_iter=30000,
                                                                                penalty='l1')))]),
             n_jobs=-1,
             param_grid={'svm__estimator__C': array([1.00000000e-03, 5.99484250e-03, 3.59381366e-02, 2.15443469e-01,
       1.29154967e+00, 7.74263683e+00, 4.64158883e+01, 2.78255940e+02,
       1.66810054e+03, 1.00000000e+04])},
             scoring='accuracy')

In [29]:
best_smote_linear_svc_species_model = smote_linear_svc_species_model.best_estimator_
y_pred_species = best_smote_linear_svc_species_model.predict(X_test)

print("Best Parameters",smote_linear_svc_species_model.best_params_)
print("Best CV Accuracy", smote_linear_svc_species_model.best_score_)

Best Parameters {'svm__estimator__C': np.float64(1.291549665014884)}
Best CV Accuracy 0.9583021711003818


In [30]:
y_pred = pd.DataFrame({'Family': y_pred_family, 'Genus':y_pred_genus, 'Species' : y_pred_species})

exact_match_score = (y_test.values == y_pred.values).all(axis = 1).mean()
hamming_loss_value = (y_test.values != y_pred.values).mean()
hamming_score = 1 - hamming_loss_value

print("Exact Match Score:", exact_match_score)
print("Hamming Loss:", hamming_loss_value)
print("Hamming Score:", hamming_score)

Exact Match Score: 0.8527095877721167
Hamming Loss: 0.0771962328238382
Hamming Score: 0.9228037671761617


#### Classifier Conclusions

Out of the three classifiers, the L1 Penalized SVM with SMOTE had the worst exact match score and hamming score at 0.8527095877721167 and 0.9228037671761617. The L1 Penalized SVM without SMOTE had a better exact match score and hamming score of 0.9124594719777674 and 0.9430291801760075. The SVM with Gaussian kernel performed the best with exact match score and hamming score of 0.9856415006947661 and 0.9893469198703103. This suggests Gaussian kernel had the advantage by being able to fit a curved decision boundary that a linear SVM cannot. The L1 Penalty can drive coefficients to 0, which can limit the linear model. Finally, SMOTE is used to balance the minority classes by oversampling the minority class. As a result, the decision boundary is shifted, which means it may have better performance on the minority classes at the expense of the majority classes. This may contribute to lower scores on the imbalanced test set.

## 2. K-Means Clustering on a Multi-Class and Multi-Label Data Set

In [31]:
X = data.drop(columns = {'Family', 'Genus', 'Species'})
y = data[['Family', 'Genus', 'Species']]

### (a) Use k-means clustering

In [32]:
silhouettte_score_list = []
k_list = np.arange(2,51,1)
for k in k_list:
    kmeans = KMeans(n_clusters = k).fit(X)
    labels = kmeans.labels_
    sh_score = silhouette_score(X, labels)
    silhouettte_score_list.append(sh_score)
best_score = max(silhouettte_score_list)
best_k = k_list[int(np.argmax(silhouettte_score_list))]
print(best_k)

5


I used silhouette score to determine the best k. When k = 1, silhouette score is underfined because there is no cluster to compare with. 

### (b) Determine which family is the majority

In [33]:
kmeans = KMeans(n_clusters = best_k).fit(X)
labels = kmeans.labels_

In [34]:
cluster_majority = {}
for c in np.unique(labels):
    members = y[labels == c]
    cluster_majority[c] = {}                    
    for col in ['Family', 'Genus', 'Species']:
        cluster_majority[c][col] = members[col].value_counts().idxmax() 
cluster_majority

{np.int32(0): {'Family': 'Hylidae',
  'Genus': 'Hypsiboas',
  'Species': 'HypsiboasCordobae'},
 np.int32(1): {'Family': 'Leptodactylidae',
  'Genus': 'Adenomera',
  'Species': 'AdenomeraHylaedactylus'},
 np.int32(2): {'Family': 'Leptodactylidae',
  'Genus': 'Adenomera',
  'Species': 'AdenomeraHylaedactylus'},
 np.int32(3): {'Family': 'Hylidae',
  'Genus': 'Hypsiboas',
  'Species': 'HypsiboasCinerascens'},
 np.int32(4): {'Family': 'Leptodactylidae',
  'Genus': 'Adenomera',
  'Species': 'AdenomeraHylaedactylus'}}

### (c) Calculate the average Hamming distance, Hamming score, and Hamming loss

In [35]:
pred = pd.DataFrame([cluster_majority[c] for c in labels])
hamming_distance = (y.values != pred.values).sum(axis=1).mean()
hamming_loss_value = (y.values != pred.values).mean()
hamming_score = 1 - hamming_loss_value
print("Hamming Distance:", hamming_distance)
print("Hamming Loss Value:", hamming_loss_value)
print("Hamming Score:", hamming_score)

Hamming Distance: 0.8642112578179291
Hamming Loss Value: 0.28807041927264304
Hamming Score: 0.7119295807273569


### Monte Carlo Simulation

In [38]:
hamming_distance_list = []
hamming_score_list = []
hamming_loss_list = []
for procedure in np.arange(1, 51, 1):
    silhouettte_score_list = []
    k_list = np.arange(2,51,1)
    for k in k_list:
        kmeans = KMeans(n_clusters = k).fit(X)
        labels = kmeans.labels_
        sh_score = silhouette_score(X, labels)
        silhouettte_score_list.append(sh_score)
    best_score = max(silhouettte_score_list)
    best_k = k_list[int(np.argmax(silhouettte_score_list))]

    #best kmeans
    kmeans = KMeans(n_clusters = best_k).fit(X)
    labels = kmeans.labels_

    #true labels
    cluster_majority = {}
    for c in np.unique(labels):
        members = y[labels == c]
        cluster_majority[c] = {}                    
        for col in ['Family', 'Genus', 'Species']:
            cluster_majority[c][col] = members[col].value_counts().idxmax() 
            
    #calculate hamming distance, loss, and score
    pred = pd.DataFrame([cluster_majority[c] for c in labels])
    hamming_distance = (y.values != pred.values).sum(axis=1).mean()
    hamming_loss_value = (y.values != pred.values).mean()
    hamming_score = 1 - hamming_loss_value
    hamming_distance_list.append(hamming_distance)
    hamming_loss_list.append(hamming_loss_value)
    hamming_score_list.append(hamming_score)

In [40]:
print('Hamming Distance Average', np.mean(hamming_distance_list))
print('Hamming Distance Standard Deviation', np.std(hamming_distance_list))

Hamming Distance Average 0.6700125086865879
Hamming Distance Standard Deviation 0.13416620826997153


## 3. ISLR 12.6.2

![My solution](../images/parta.png)

![My solution](../images/partb.png)

![My solution](../images/partcde.png)